# Week 11, Day 5: Raising errors and debugging

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w011d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def average(values):
    if len(values) == 0:
        return 0
    return sum(values) / len(values)

print(average([4, 6]))
print(average([]))

In [ ]:
def average(values):
    if len(values) == 0:
        raise ValueError("can't average an empty list")
    return sum(values) / len(values)

print(average([4, 6]))
print(average([]))

In [ ]:
def average(values):
    if len(values) == 0:
        raise ValueError("can't average an empty list")
    return sum(values) / len(values)

for group in [[90, 70], []]:
    try:
        print("Average:", average(group))
    except ValueError as e:
        print("No average:", e)

> **Key idea.** When a function is asked to do something impossible or meaningless, `raise` an exception with a message that says what was wrong and, where it helps, the bad value. Let the caller decide what to do about it.

In [ ]:
def parse_time(text):
    """Turn 'HH:MM' into minutes after midnight."""
    parts = text.split(":")
    if len(parts) != 2:
        raise ValueError(f"expected HH:MM, got {text!r}")
    hours, minutes = int(parts[0]), int(parts[1])
    if not (0 <= hours < 24 and 0 <= minutes < 60):
        raise ValueError(f"no such time: {text!r}")
    return hours * 60 + minutes

for t in ["09:30", "9.30", "25:00", "ab:cd"]:
    try:
        print(t, "->", parse_time(t))
    except ValueError as e:
        print(t, "->", e)

In [ ]:
def load_number(text, log):
    try:
        return int(text)
    except ValueError:
        log.append(f"bad number: {text!r}")
        raise

log = []
try:
    load_number("12", log)
    load_number("twelve", log)
finally:
    print("log:", log)

In [ ]:
def split_bill(total, people):
    shares = [round(total / people, 2)] * people
    shares[-1] = round(total - sum(shares[:-1]), 2)
    assert round(sum(shares), 2) == total, "shares don't add up"
    return shares

print(split_bill(100, 3))

> **Careful.** Don't use `assert` to check what users type or what's in a file. Python can be run with assertions switched off, and then those checks vanish. For input, use `if` and `raise` (or a validation loop).

*Interactive figure on the website: Choose a program and click the parts of the error. Each File line is one step in the chain of calls; the last one is where it actually went wrong..*

In [ ]:
def count_vowels(text):
    count = 0
    for ch in text:
        print(f"{ch=} {count=}")
        if ch in "aeiou":
            count += 1
    return count

print(count_vowels("Ada"))

> **Tip.** When you fix a bug, ask: what other inputs would have shown it? Try them. Bugs come in families, and the same mistake is often waiting somewhere else in your code.

### Exercise: Valid ages only

Write `check_age(age)` that returns `age` if it's an int from 0 to 130, raises a `TypeError` with the message `age must be a whole number` if it isn't an int, and raises a `ValueError` with the message `age must be 0 to 130, got N` if it's out of range.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def raises(fn, kind, msg):\n    try:\n        fn()\n    except kind as e:\n        assert str(e) == msg, f\"Message should be {msg!r}, got {str(e)!r}\"\n        return\n    raise AssertionError(f\"Expected a {kind.__name__}\")\ndef test_values():\n    \"good and bad ages\"\n    assert check_age(30) == 30 and check_age(0) == 0 and check_age(130) == 130\n    raises(lambda: check_age(-5), ValueError, \"age must be 0 to 130, got -5\")\n    raises(lambda: check_age(131), ValueError, \"age must be 0 to 130, got 131\")\n    raises(lambda: check_age(\"30\"), TypeError, \"age must be a whole number\")\n    raises(lambda: check_age(30.5), TypeError, \"age must be a whole number\")")

### Exercise: No overdrafts

Write `withdraw(balance, amount)` that returns the new balance. Raise a `ValueError` if `amount` isn't positive (message `amount must be positive`) or is more than the balance (message `insufficient funds: balance B, asked for A`).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def raises(fn, msg):\n    try:\n        fn()\n    except ValueError as e:\n        assert str(e) == msg, f\"Message should be {msg!r}, got {str(e)!r}\"\n        return\n    raise AssertionError(\"Expected a ValueError\")\ndef test_values():\n    \"withdrawals\"\n    assert withdraw(500, 200) == 300\n    assert withdraw(500, 500) == 0\n    raises(lambda: withdraw(500, 0), \"amount must be positive\")\n    raises(lambda: withdraw(500, -1), \"amount must be positive\")\n    raises(lambda: withdraw(300, 400), \"insufficient funds: balance 300, asked for 400\")")

### Exercise: Handle it in the caller

Using your `withdraw` from the starter, write a program that processes `requests` from a balance of 1000, printing `Paid A, balance B` after each successful withdrawal and `Refused: MESSAGE` for each one that raises.

In [ ]:
def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError("amount must be positive")
    if amount > balance:
        raise ValueError(f"insufficient funds: balance {balance}, asked for {amount}")
    return balance - amount

requests = [300, 0, 900, 500]
balance = 1000

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the run\"\n    expect_output(\"Paid 300, balance 700\\nRefused: amount must be positive\\nRefused: insufficient funds: balance 700, asked for 900\\nPaid 500, balance 200\")")

### Exercise: A real date?

Write `parse_date(text)` that turns `"DD/MM/YYYY"` into a tuple `(year, month, day)` of ints. Raise a `ValueError` with the message `not a date: TEXT` (with the text in quotes, as `{text!r}` gives) if the format is wrong or the date doesn't exist. February has 29 days in a leap year (divisible by 4, except centuries not divisible by 400) and 28 otherwise.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def bad(t):\n    try:\n        parse_date(t)\n    except ValueError as e:\n        assert str(e) == f\"not a date: {t!r}\", f\"Message for {t!r} should be {'not a date: ' + repr(t)!r}, got {str(e)!r}\"\n        return\n    raise AssertionError(f\"{t!r} should raise a ValueError\")\ndef test_good():\n    \"real dates\"\n    assert parse_date(\"27/10/2026\") == (2026, 10, 27)\n    assert parse_date(\"29/02/2024\") == (2024, 2, 29)\n    assert parse_date(\"29/02/2000\") == (2000, 2, 29)\n    assert parse_date(\"31/12/1999\") == (1999, 12, 31)\ndef test_bad():\n    \"not dates\"\n    for t in [\"29/02/2026\", \"29/02/1900\", \"31/04/2026\", \"00/01/2026\", \"1/13/2026\", \"27-10-2026\", \"aa/bb/cccc\", \"\"]:\n        bad(t)")

### Exercise: Find it with prints

`count_capitals` should count the capital letters, but `count_capitals("Python Atlas")` gives 12. There are two bugs, and fixing the first makes the answer 0. Use a debugging print to find each one, fix them, and remove the print.

In [ ]:
def count_capitals(text):
    count = 0
    for ch in text:
        if ch.isupper:
            count += 1
        else:
            count = 0
    return count

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert count_capitals(\"Python Atlas\") == 2\n    assert count_capitals(\"ABC def G\") == 4\n    assert count_capitals(\"\") == 0\n    assert \"print\" not in source, \"Remove your debugging print.\"")

### Bug hunt: The error that got lost

`safe_load` should return the numbers in a file, or raise a `ValueError` saying which line was bad, like `line 2: 'x'`. Instead, for a bad file it returns an empty list, so the problem goes unnoticed. Fix it.

In [ ]:
def safe_load(name):
    numbers = []
    with open(name) as f:
        for n, line in enumerate(f, start=1):
            try:
                numbers.append(float(line))
            except ValueError:
                return []
    return numbers

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"good and bad files\"\n    write_file(\"good.txt\", \"1\\n2.5\\n\")\n    write_file(\"bad.txt\", \"1\\nx\\n3\\n\")\n    assert safe_load(\"good.txt\") == [1.0, 2.5]\n    try:\n        safe_load(\"bad.txt\")\n    except ValueError as e:\n        assert str(e) == \"line 2: 'x'\", f\"Message should be \\\"line 2: 'x'\\\", got {str(e)!r}\"\n    else:\n        raise AssertionError(\"A bad line should raise a ValueError\")")

### Bug hunt: Off by one, somewhere

`moving_average(values, k)` should return the averages of each run of `k` neighbouring values: `moving_average([1, 2, 3, 4], 2)` is `[1.5, 2.5, 3.5]`. It returns one too few. Reproduce it with the smallest input you can, find the bug, and fix it.

In [ ]:
def moving_average(values, k):
    result = []
    for start in range(len(values) - k):
        window = values[start:start + k]
        result.append(sum(window) / k)
    return result

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert moving_average([1, 2, 3, 4], 2) == [1.5, 2.5, 3.5]\n    assert moving_average([5], 1) == [5.0]\n    assert moving_average([1, 2, 3], 3) == [2.0]\n    assert moving_average([1, 2], 3) == []")

### Bug hunt: assert for user input

This checks the user's quantity with `assert`, which can be switched off. Replace the assertion with an `if` that raises a `ValueError` with the same message.

In [ ]:
def order(qty):
    assert qty > 0, "quantity must be positive"
    return qty * 950

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"checked properly\"\n    assert order(2) == 1900\n    try:\n        order(0)\n    except ValueError as e:\n        assert str(e) == \"quantity must be positive\"\n    else:\n        raise AssertionError(\"order(0) should raise a ValueError\")\n    assert \"assert\" not in source")

### Puzzle: Log and pass it on

Arrange the lines so that `checked_int` records a bad value in `problems` and then lets the error carry on. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work out what each program prints, as a string with `\n` between lines.

In [ ]:
# Program A
#   def f(x):
#       if x < 0:
#           raise ValueError("negative")
#       return x * 2
#   try:
#       print(f(3))
#       print(f(-1))
#       print(f(5))
#   except ValueError as e:
#       print("caught", e)
a = ___
# Program B
#   x = 5
#   print(f"{x=}", f"{x * 2=}")
b = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each output\"\n    assert a == \"6\\ncaught negative\"\n    assert b == \"x=5 x * 2=10\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a strict config reader

Programs often read settings from a simple file of `key = value` lines, where `#` starts a comment and blank lines are ignored. Write `read_config(name, schema)`, where `schema` maps each allowed key to its type (`int`, `float`, `bool` or `str`), that returns a dictionary of settings, converted to their types. Raise a `ValueError` whose message starts with the file name and line number, as in `app.cfg line 3: ...`, for:

- a line without `=`: `expected key = value`;
- an unknown key: `unknown key 'colour'`;
- a key given twice: `duplicate key 'size'`;
- a value that can't be converted: `'big' is not a valid int`. A bool must be `true` or `false` (any case).

And raise a `ValueError` `app.cfg: missing keys: a, b` (sorted, after reading every line) if any key in the schema wasn't given.

In [ ]:
# Run this cell first: it creates the files used below.
with open('app.cfg', 'w', encoding='utf-8') as f:
    f.write('# window settings\ntitle = Python Atlas\nsize = 14\n\ndark = TRUE   \nratio = 1.5\n')

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("SCHEMA = {\"title\": str, \"size\": int, \"dark\": bool, \"ratio\": float}\ndef err(text, schema=SCHEMA):\n    write_file(\"t.cfg\", text)\n    try:\n        read_config(\"t.cfg\", schema)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(\"Expected a ValueError for:\\n\" + text)\ndef test_good():\n    \"the example file\"\n    assert read_config(\"app.cfg\", SCHEMA) == {\"title\": \"Python Atlas\", \"size\": 14, \"dark\": True, \"ratio\": 1.5}\ndef test_errors():\n    \"the errors\"\n    assert err(\"title = a\\nsize 14\\n\") == \"t.cfg line 2: expected key = value\"\n    assert err(\"colour = red\\n\") == \"t.cfg line 1: unknown key 'colour'\"\n    assert err(\"size = 1\\nsize = 2\\n\") == \"t.cfg line 2: duplicate key 'size'\"\n    assert err(\"# x\\nsize = big\\n\") == \"t.cfg line 2: 'big' is not a valid int\"\n    assert err(\"dark = yes\\n\") == \"t.cfg line 1: 'yes' is not a valid bool\"\n    assert err(\"title = x\\nsize = 2\\n\") == \"t.cfg: missing keys: dark, ratio\"\ndef test_equals_in_value():\n    \"an = inside the value\"\n    write_file(\"e.cfg\", \"title = a = b\\n\")\n    assert read_config(\"e.cfg\", {\"title\": str}) == {\"title\": \"a = b\"}")

## Warm-up

`average([])` returned 0. Is that right? An empty class doesn't have an average mark of 0: it has **no** average. A program that uses that 0 might report that the class failed. Sometimes the honest answer is "this can't be done", and today you'll learn to give it.

## Raising exceptions

The `raise` statement raises an exception, exactly as Python does when something goes wrong:

`raise ValueError("...")` makes a `ValueError` with your message and raises it. The function stops at once, like `return`, but instead of handing back a value it hands back a problem. The caller can catch it, or let it travel further up.

### Why raise instead of returning something?

Returning a special value like `0`, `-1` or `None` means every caller has to remember to check for it, and a caller that forgets carries on with a wrong value. An exception **can't be ignored by accident**: either something handles it, or the program stops and says exactly where and why.

### Which type?

Use the built-in type that fits:

- `ValueError`: the argument has the right type but a bad value (a negative age, an empty list, a date of 31 February);
- `TypeError`: the argument is the wrong type altogether;
- `KeyError`, `IndexError`: something asked for isn't there;
- `FileNotFoundError`: a needed file is missing.

Put the bad value in the message. `ValueError("age must be 0 to 130, got -5")` tells the reader far more than `ValueError("bad age")`.

Notice the last one. `int("ab")` raised its own `ValueError`, with Python's message, which also travels up to the same `except`. `{text!r}` in an f-string shows the value with its quotes, so an empty string or stray spaces are visible.

### Raise early

Check the inputs at the **start** of a function, and raise before doing any work. Then nothing is half-done when the error happens, and the error points at the real cause rather than at some later line that tripped over the bad value.

## Re-raising and assert

### Re-raising

Sometimes you want to notice an error, do something, and still let it carry on up. A bare `raise` inside an `except` block re-raises the exception being handled:

### assert

`assert condition, message` raises an `AssertionError` if the condition is false, and does nothing if it's true. It's for checking **your own assumptions**, things that should be true if your code is right:

If the assertion ever fails, you've found a bug in **your** code, right where your belief stopped being true. You've used `assert` in tests all along: that's exactly what the checks on this site do.

## Tracebacks through functions

When an error happens several function calls deep, the traceback lists every call in the chain, **outermost first**, ending with the line that actually failed:

Read it from the **bottom**: the last two lines say what went wrong and where. If the cause isn't there, move **up** one step at a time. In the first program, `int` failed on `"4x"` inside `to_minutes`, but the bad value came from the list in the last line of the program. The traceback shows you the path from one to the other.

## A method for debugging

A **bug** is a difference between what you meant and what you wrote. Debugging well isn't about being clever; it's about being systematic. Here's a method that works on any bug:

1. **Reproduce it.** Find an input that makes it happen every time. The smaller the input, the better.
2. **Read the evidence.** The error message, the traceback, or the exact difference between the output you got and the output you expected.
3. **Locate it.** Find the first place where the program's values differ from what you expect.
4. **Guess the cause, then test the guess.** Make one specific prediction ("`total` is still 0 after the loop, because the `+=` is outside it") and check it.
5. **Fix it, and check the fix**, with the original input and a few others.

The most common mistake is skipping straight to step 5: changing things at random until the problem seems to go away. That often hides bugs instead of fixing them.

### Locating: print the values

The simplest tool is still `print`. The f-string form `f"{name=}"` prints a name **and** its value, which is perfect for debugging:

The prints make the problem visible straight away: the capital `A` isn't in `"aeiou"`. Remove debugging prints once you've found the bug.

### Locating: halve the search

When you don't know where a long program goes wrong, check the values **halfway** through. If they're already wrong, the bug is in the first half; if they're right, it's in the second. Then halve that half. Ten checks are enough to find one bad line among a thousand. You can do the same with input data: if a 1,000-line file breaks your program, try the first 500 lines.

### Explain it out loud

Explaining your code, line by line, to someone else, or even to a toy duck on your desk, often reveals the bug before you finish, because explaining forces you to say what each line **actually** does rather than what you meant. Programmers call this **rubber duck debugging**, and it really works. The tutor on this site is a good listener too.

## Debugging tools

- **X-ray** on this site runs a program one step at a time and shows every variable. Use it when you can't tell which line goes wrong.
- **Trace tables** on paper (Week 3) do the same with a pencil, and make you think about every line.
- **Debuggers** in code editors such as VS Code and Thonny let you pause a program on a chosen line (a **breakpoint**) and inspect everything. You'll set one up in Phase 2.
- **Tests**, small checks that your functions give the right answers, catch bugs before they reach anyone else. You'll write your own in Phase 2.

## Practice

## Quiz

## Daily challenge

## Before you go

Next time a program of yours goes wrong, before changing anything, write down the five steps of the method and follow them. Then find one function from an earlier week that returns a special value for bad input, and decide whether it should raise instead. Tomorrow is the Forge: a program that reads messy real-world data files, survives every problem in them, and reports exactly what it found.